# 05 — PackNet: Structured Pruning for Task Isolation

PackNet (Mallya & Lazebnik, CVPR 2018) takes a purely architectural approach: after training each task,
it **prunes** the fraction of *free* weights with smallest absolute value and **freezes** the surviving
free weights as the task's sub-network.  Future tasks can only use the remaining free capacity.

This creates perfectly non-overlapping sub-networks — one per task — so there is zero interference
between tasks and BWT ≈ 0.  The trade-off is a fixed total network capacity shared across all tasks.

**Key difference from CausalPruner**: PackNet selects weights to protect by *magnitude* (largest |w|),
which is purely correlational.  CausalPruner selects by *Fisher score* (causal importance).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.train import run_continual
from src.evaluate import backward_transfer, forward_transfer, average_accuracy, print_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Run PackNet

In [ ]:
R_pn = run_continual(
    method='packnet',
    dataset='cifar100',
    n_tasks=20,
    epochs_per_task=10,
    lr=0.1,
    batch_size=128,
    device=device,
    prune_ratio=0.5,   # prune 50% of free weights per task
    seed=42,
    save_dir='../results/improved',
    verbose=True,
)

## 2. Accuracy Matrix and Forgetting

In [ ]:
T = R_pn.shape[0]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Heatmap
display = np.where(np.isnan(R_pn), 0.0, R_pn * 100)
im = axes[0].imshow(display, vmin=0, vmax=100, cmap='Greens', aspect='auto')
plt.colorbar(im, ax=axes[0], label='Accuracy (%)')
axes[0].set_xticks(range(T)); axes[0].set_yticks(range(T))
axes[0].set_xticklabels([f'T{j}' for j in range(T)], fontsize=7)
axes[0].set_yticklabels([f'T{i}' for i in range(T)], fontsize=7)
axes[0].set_xlabel('Task evaluated'); axes[0].set_ylabel('After training task')
axes[0].set_title('PackNet — Accuracy Matrix', fontweight='bold')

# Per-task final accuracy
final = R_pn[T - 1, :] * 100
axes[1].bar(range(T), final, color='#2ecc71', alpha=0.8)
axes[1].set_xlabel('Task index'); axes[1].set_ylabel('Final accuracy (%)')
axes[1].set_title('PackNet — Per-task Final Accuracy', fontweight='bold')
axes[1].set_ylim(0, 105); axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/improved/packnet_results.png', bbox_inches='tight')
plt.show()

## 3. Sparsity Accumulation Over Tasks

In [ ]:
# Theoretical sparsity: after task t, fraction protected = sum_{k=0}^{t} (0.5)^{k+1}
sparsity_per_task = []
free = 1.0
for t in range(20):
    protected_this_task = free * 0.5
    free -= protected_this_task
    sparsity_per_task.append(1.0 - free)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(20), [s * 100 for s in sparsity_per_task],
        marker='o', color='#2ecc71', linewidth=2)
ax.fill_between(range(20), [s * 100 for s in sparsity_per_task], alpha=0.2, color='#2ecc71')
ax.set_xlabel('Task index'); ax.set_ylabel('Network sparsity (%)')
ax.set_title('PackNet: Cumulative Sparsity After Each Task (prune_ratio=0.5)',
             fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('../results/improved/packnet_sparsity.png', bbox_inches='tight')
plt.show()

## 4. Prune Ratio Ablation

In [ ]:
prune_ratios = [0.3, 0.4, 0.5, 0.6, 0.7]
pn_avgs, pn_bwts = [], []

for pr in prune_ratios:
    print(f'  prune_ratio={pr} ...', end='')
    R = run_continual(
        method='packnet', dataset='cifar100', n_tasks=5,
        epochs_per_task=5, lr=0.1, batch_size=128,
        device=device, prune_ratio=pr, seed=42,
        save_dir='../results/improved', verbose=False,
    )
    pn_avgs.append(average_accuracy(R) * 100)
    pn_bwts.append(backward_transfer(R) * 100)
    print(f' avg_acc={pn_avgs[-1]:.1f}%  bwt={pn_bwts[-1]:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(prune_ratios, pn_avgs, marker='o', color='#2ecc71', linewidth=2)
axes[0].set_xlabel('Prune ratio'); axes[0].set_ylabel('Avg Accuracy (%)')
axes[0].set_title('PackNet: Accuracy vs Prune Ratio'); axes[0].grid(True, alpha=0.3)

axes[1].plot(prune_ratios, pn_bwts, marker='s', color='#e74c3c', linewidth=2)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Prune ratio'); axes[1].set_ylabel('BWT (%)')
axes[1].set_title('PackNet: BWT vs Prune Ratio'); axes[1].grid(True, alpha=0.3)

plt.suptitle('PackNet Prune Ratio Ablation (5 tasks)', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/improved/packnet_prune_ablation.png', bbox_inches='tight')
plt.show()

## 5. Metrics

In [ ]:
print_metrics(R_pn, method='PackNet (prune_ratio=0.5)')